<a href="https://colab.research.google.com/github/AngeloSorte/Financial-Risk-Insights-using-Live-API-Data/blob/angelosorte.github.io/Credit_Risk_Analysis_using_API_Data_(World_Bank_%2B_Machine_Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ====== IMPORT ======
import requests
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ====== FETCH DATA FROM WORLD BANK API ======
# GDP per capita (indicator: NY.GDP.PCAP.CD)
url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=10000"

response = requests.get(url)
data = response.json()

# Extract data (skip metadata)
records = data[1]

# Convert to DataFrame
df = pd.DataFrame(records)

# ====== CLEAN DATA ======
# Keep relevant columns
df = df[['country', 'date', 'value']]

# Extract country name
df['country'] = df['country'].apply(lambda x: x['value'])

# Rename columns
df.columns = ['country', 'year', 'gdp_per_capita']

# Remove missing values
df = df.dropna()

# Convert year to int
df['year'] = df['year'].astype(int)

# Keep only recent years
df = df[df['year'] >= 2015]

# ====== FEATURE ENGINEERING ======
# Create a binary risk label (simple heuristic)
# Low GDP per capita → higher risk
df['risk'] = df['gdp_per_capita'].apply(lambda x: 1 if x < 10000 else 0)

# Features and target
X = df[['gdp_per_capita']]
y = df['risk']

# ====== TRAIN TEST SPLIT ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ====== SCALING ======
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====== MODEL ======
model = LogisticRegression()
model.fit(X_train, y_train)

# ====== EVALUATION ======
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# ====== SAMPLE PREDICTION ======
sample = [[5000]]  # Example GDP per capita
sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled)

print("\nSample prediction (GDP=5000):", "High Risk" if prediction[0] == 1 else "Low Risk")

Accuracy: 0.9727520435967303
              precision    recall  f1-score   support

           0       1.00      0.93      0.96       136
           1       0.96      1.00      0.98       231

    accuracy                           0.97       367
   macro avg       0.98      0.96      0.97       367
weighted avg       0.97      0.97      0.97       367


Sample prediction (GDP=5000): High Risk


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
